# Notebook 2: Persistent Conversational Agent with LangGraph

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain the **Dual-Track Memory** pattern for conversational agents.
2. Implement **persistent user profiles** that survive across sessions.
3. Build a **smart update function** that merges new data without overwriting existing data.
4. Understand how to pass **extra state fields** (like `client_id`) through a LangGraph agent.
5. Run a working **Financial Advisor Agent** that remembers client preferences.

---

## Use Case: Personalized Financial Advisor

> **Scenario**: A financial advisory platform serves multiple clients. Each client has a profile storing their risk tolerance, investment goals, and portfolio interests. When a client chats with the advisor bot, it should:
> 1. **Retrieve** their existing profile to give personalized advice.
> 2. **Listen** for any new preferences mentioned in the conversation.
> 3. **Update** the profile intelligently (appending to lists, updating scalars).

---

## The Dual-Track Memory Architecture

```
┌──────────────────────────────────────────────────────────────────┐
│                        USER MESSAGE                              │
│  "What are my interests? Also, I'm now interested in AI stocks." │
└────────────────────────────┬─────────────────────────────────────┘
                             │
                             ▼
┌──────────────────────────────────────────────────────────────────┐
│                     REASONER NODE                                │
│                                                                  │
│  Track 1 (Retrieve): "I should check the client's profile"      │
│  → Calls: get_client_profile(client_id="client_123")            │
│                                                                  │
│  Track 2 (Capture): "User mentioned AI stocks"                  │
│  → Calls: extract_financial_details(portfolio_interests=[...])  │
└────────────────────────────┬─────────────────────────────────────┘
                             │
                             ▼
┌──────────────────────────────────────────────────────────────────┐
│                       TOOL NODE                                  │
│                                                                  │
│  get_client_profile → Returns existing profile JSON             │
│  extract_financial_details → Triggers smart_update()            │
│                           → Profile updated in database         │
└────────────────────────────┬─────────────────────────────────────┘
                             │
                             ▼
┌──────────────────────────────────────────────────────────────────┐
│              REASONER NODE (Final Response)                      │
│                                                                  │
│  "Your interests are ETFs, bonds, and now AI stocks!"           │
└──────────────────────────────────────────────────────────────────┘
```

---

## Key Concepts

| Concept | Description |
|---|---|
| **Dual-Track** | Agent simultaneously retrieves existing data AND captures new data in one turn |
| **Smart Update** | Merges updates: appends to lists, overwrites scalars — no data loss |
| **Extra State Fields** | `client_id` is passed in the state so all nodes know which user they're serving |
| **Tool Interception** | The `tool_node` inspects which tool was called and triggers side effects accordingly |

## ⚙️ Step 1: Setup and Azure OpenAI Configuration

In [1]:
# Install required packages (uncomment if needed)
# !pip install langgraph langchain-openai langchain

import os
import json
from dotenv import load_dotenv
from typing import TypedDict, Sequence, Annotated
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, SystemMessage, ToolMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END

# ── Load credentials from .env ─────────────────────────────────────────────────
load_dotenv()

model = AzureChatOpenAI(azure_deployment="gpt-4.1", temperature=0)
print("Azure OpenAI model initialized!")

Azure OpenAI model initialized!


## 🗄️ Step 2: The Persistent Memory Store

In production, this would be a PostgreSQL or MongoDB database. Here we use a Python dictionary to keep things simple and focus on the agent logic.

In [2]:
# ── Mock Database ────────────────────────────────────────────────────────────
# In production: replace with PostgreSQL, MongoDB, or a vector store
client_profiles = {
    "client_123": {
        "name": "Joseph",
        "risk_tolerance": "moderate",
        "investment_goals": "retirement planning",
        "portfolio_interests": ["ETFs", "bonds"],
        "preferred_sectors": ["technology", "healthcare"]
    },
    "client_456": {
        "name": "Sarah",
        "risk_tolerance": "high",
        "investment_goals": "wealth accumulation",
        "portfolio_interests": ["growth stocks", "crypto"],
        "preferred_sectors": ["fintech", "energy"]
    }
}

print("✅ Client profiles loaded:")
for cid, profile in client_profiles.items():
    print(f"  {cid}: {profile['name']} | Risk: {profile['risk_tolerance']} | Interests: {profile['portfolio_interests']}")

✅ Client profiles loaded:
  client_123: Joseph | Risk: moderate | Interests: ['ETFs', 'bonds']
  client_456: Sarah | Risk: high | Interests: ['growth stocks', 'crypto']


## 🛠️ Step 3: Defining the Memory Tools

We define two tools:
- **`get_client_profile`**: Read-only — retrieves the current profile.
- **`extract_financial_details`**: Write-trigger — the agent calls this when it detects new information in the conversation. The tool itself just returns the data; the actual database update happens in our `tool_node` via `smart_update`.

> **Design Pattern**: The extraction tool is a "signal" to the system that new data was found. The actual persistence logic lives in the tool node, not the tool itself.

In [3]:
@tool
def get_client_profile(client_id: str) -> str:
    """Retrieve the stored financial profile for a client.
    Use this at the start of every conversation to personalize the response.
    Returns a JSON string with the client's risk tolerance, goals, and interests.
    """
    profile = client_profiles.get(client_id, {})
    if profile:
        return json.dumps(profile, indent=2)
    return "No profile found for this client."

@tool
def extract_financial_details(
    risk_tolerance: str = None,
    investment_goals: str = None,
    portfolio_interests: list = None,
    preferred_sectors: list = None
) -> str:
    """Extract and save new financial preferences mentioned by the client.
    Call this whenever the client mentions new investment interests, changes in risk tolerance,
    new financial goals, or preferred market sectors.
    Only include fields that were explicitly mentioned by the client.
    """
    # Collect only the fields that were actually provided
    updates = {}
    if risk_tolerance: updates["risk_tolerance"] = risk_tolerance
    if investment_goals: updates["investment_goals"] = investment_goals
    if portfolio_interests: updates["portfolio_interests"] = portfolio_interests
    if preferred_sectors: updates["preferred_sectors"] = preferred_sectors
    return json.dumps(updates)

tools = [get_client_profile, extract_financial_details]
tools_by_name = {t.name: t for t in tools}
model_with_tools = model.bind_tools(tools)

print("✅ Memory tools created and bound to model.")

✅ Memory tools created and bound to model.


## 🧠 Step 4: The Smart Update Function

This is the heart of the persistent memory system. Instead of blindly overwriting the profile, we:
- **Append** to lists (so "AI stocks" gets added to existing interests, not replace them).
- **Overwrite** scalars (risk tolerance changes completely, not accumulates).
- **Add** new fields if they didn't exist before.

In [4]:
def smart_update(client_id: str, updates: dict):
    """Intelligently merge updates into the existing client profile.
    
    Rules:
    - List fields: EXTEND (no duplicates) — preserves history
    - Scalar fields: OVERWRITE — reflects current state
    - New fields: ADD — expands the profile
    """
    if client_id not in client_profiles:
        client_profiles[client_id] = {}
        
    profile = client_profiles[client_id]
    changed_fields = []
    
    for key, new_value in updates.items():
        if key in profile and isinstance(profile[key], list) and isinstance(new_value, list):
            # Extend lists without duplicates
            before = set(profile[key])
            profile[key] = list(set(profile[key] + new_value))
            added = set(profile[key]) - before
            if added:
                changed_fields.append(f"{key}: added {list(added)}")
        else:
            old_value = profile.get(key, "N/A")
            profile[key] = new_value
            changed_fields.append(f"{key}: '{old_value}' → '{new_value}'")
    
    if changed_fields:
        print(f"  💾 Profile updated for {client_id}:")
        for change in changed_fields:
            print(f"     • {change}")

print("✅ smart_update function defined.")

✅ smart_update function defined.


## 🏗️ Step 5: Building the LangGraph Agent

We add `client_id` as an extra field in the state so all nodes know which client they're serving.

In [5]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    client_id: str  # Extra field: which client is this conversation for?

def call_model(state: AgentState):
    system_prompt = SystemMessage(
        content=(
            f"You are a personalized Financial Advisor. "
            f"The current client ID is: {state['client_id']}. "
            f"ALWAYS start by calling get_client_profile to retrieve their profile. "
            f"If the client mentions new investment preferences, risk tolerance changes, or new goals, "
            f"call extract_financial_details to save them. "
            f"Then provide a warm, personalized response using their profile information."
        )
    )
    response = model_with_tools.invoke([system_prompt] + list(state["messages"]))
    return {"messages": [response]}

def tool_node(state: AgentState):
    last_message = state["messages"][-1]
    results = []
    
    for tc in last_message.tool_calls:
        print(f"  🛠️ Calling tool: {tc['name']}")
        tool_fn = tools_by_name[tc["name"]]
        output = tool_fn.invoke(tc["args"])
        
        # Intercept extraction calls to trigger smart_update
        if tc["name"] == "extract_financial_details":
            updates = json.loads(output)
            if updates:
                smart_update(state["client_id"], updates)
            
        results.append(ToolMessage(content=str(output), tool_call_id=tc["id"]))
        
    return {"messages": results}

def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

# Build and compile
graph = StateGraph(AgentState)
graph.add_node("reasoner", call_model)
graph.add_node("tools", tool_node)
graph.set_entry_point("reasoner")
graph.add_conditional_edges("reasoner", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "reasoner")
app = graph.compile()

print("✅ Persistent Memory Agent compiled!")

✅ Persistent Memory Agent compiled!


## 🧪 Step 6: Testing the Dual-Track Memory

Let's simulate a conversation where Joseph (client_123) asks about his profile AND provides new information.

In [6]:
client_id = "client_123"
user_input = (
    "Hi! What are my current investment interests? "
    "By the way, I just got a big promotion, so my risk tolerance is now 'high'. "
    "I'm also interested in 'AI stocks' and 'semiconductor ETFs' now."
)

print(f"👤 User ({client_id}): {user_input}\n")
print("─" * 60)

initial_state = {
    "messages": [HumanMessage(content=user_input)],
    "client_id": client_id
}

for event in app.stream(initial_state):
    for node_name, node_state in event.items():
        if node_name == "reasoner":
            msg = node_state["messages"][-1]
            if not msg.tool_calls:
                print(f"\n🤖 Advisor: {msg.content}")

👤 User (client_123): Hi! What are my current investment interests? By the way, I just got a big promotion, so my risk tolerance is now 'high'. I'm also interested in 'AI stocks' and 'semiconductor ETFs' now.

────────────────────────────────────────────────────────────
  🛠️ Calling tool: get_client_profile
  🛠️ Calling tool: extract_financial_details
  💾 Profile updated for client_123:
     • risk_tolerance: 'moderate' → 'high'
     • portfolio_interests: added ['AI stocks', 'semiconductor ETFs']

🤖 Advisor: Congratulations on your promotion, Joseph! That’s fantastic news, and it’s great to see you’re ready to embrace a higher risk tolerance in your investment strategy.

Currently, your investment interests include AI stocks and semiconductor ETFs, reflecting your enthusiasm for innovative and high-growth sectors. Previously, you were interested in ETFs and bonds, with a focus on technology and healthcare. With your new risk profile, you’re well-positioned to explore more dynamic oppor

### 🔍 Verify the Profile Was Updated

In [7]:
print("\n📊 Updated Profile for client_123:")
print(json.dumps(client_profiles["client_123"], indent=2))


📊 Updated Profile for client_123:
{
  "name": "Joseph",
  "risk_tolerance": "high",
  "investment_goals": "retirement planning",
  "portfolio_interests": [
    "bonds",
    "AI stocks",
    "semiconductor ETFs",
    "ETFs"
  ],
  "preferred_sectors": [
    "technology",
    "healthcare"
  ]
}


## 🧪 Step 7: Second Turn — Testing Memory Persistence

Now let's simulate a second message from the same client. The agent should remember the updated profile.

In [8]:
user_input2 = "Based on my profile, what kind of investment strategy would you recommend for me?"

print(f"👤 User ({client_id}): {user_input2}\n")
print("─" * 60)

initial_state2 = {
    "messages": [HumanMessage(content=user_input2)],
    "client_id": client_id
}

for event in app.stream(initial_state2):
    for node_name, node_state in event.items():
        if node_name == "reasoner":
            msg = node_state["messages"][-1]
            if not msg.tool_calls:
                print(f"\n🤖 Advisor: {msg.content}")

👤 User (client_123): Based on my profile, what kind of investment strategy would you recommend for me?

────────────────────────────────────────────────────────────
  🛠️ Calling tool: get_client_profile

🤖 Advisor: Joseph, based on your profile, you have a high risk tolerance and your primary investment goal is retirement planning. You’re interested in bonds, AI stocks, semiconductor ETFs, and general ETFs, with a preference for the technology and healthcare sectors.

Given your high risk tolerance and sector preferences, I would recommend a growth-oriented investment strategy. This could include:

- Allocating a significant portion of your portfolio to technology and healthcare stocks, especially those in AI and semiconductors, to capitalize on innovation and long-term growth.
- Balancing your portfolio with ETFs for diversification, including sector-specific ETFs in technology and healthcare.
- Including some bonds to provide stability and income, even though your risk tolerance is h

## 📚 Summary and Key Takeaways

| Component | Role | Key Code |
|---|---|---|
| `client_profiles` dict | Simulates persistent database | Replace with real DB in production |
| `get_client_profile` tool | Read track — retrieves existing data | Called at start of every turn |
| `extract_financial_details` tool | Write track — captures new data | Called when new info detected |
| `smart_update()` | Merges updates intelligently | Extends lists, overwrites scalars |
| `client_id` in state | Identifies which user | Passed through all nodes |

### 🔑 Key Insights

1. **Dual-Track is Simultaneous**: The agent can retrieve AND update in the same turn — no need for separate "read" and "write" conversations.
2. **Smart Merging Preserves History**: A client's interests accumulate over time rather than being replaced.
3. **Tool Interception Pattern**: The `tool_node` is the right place to add side effects (like DB writes) triggered by tool calls.

### 🚀 Next Steps
- **Notebook 3**: Learn prompt chaining for complex multi-step analysis workflows.
